# 模块九：自动血缘采集 — OpenLineage (Phase 2 / Background §6.9)

## 学习目标

1. 理解从「手工 `lineage_recipe.yaml`」升级为「OpenLineage 自动 emit」的动机与收益。
2. 跑通 **auto 通道**（ETL 任务 emit `LineageEmitter`）并与 **manual 通道**（`emit_lineage.py`）对比。
3. 抓 DataHub UI Lineage 标签页的截图作为视觉证据。

## 与 module3 的关系

| 维度 | module3（Phase 1 演示版） | module9（Phase 2 升级） |
|---|---|---|
| 血缘来源 | 手工 `lineage_recipe.yaml`（5~8 条） | ETL 任务自动 emit（理论上 N×M 条） |
| 业务代码改动 | 无（YAML 维护） | ≤ 5 行/ETL 入口（上下文管理器） |
| 事件格式 | 直接写 `upstreamLineage` aspect | OpenLineage 1.x → GMS 转换器写 `dataJobInputOutput` |
| 跨系统 JOIN | YAML 手写 | 仍走 YAML（SQL 解析对业务键不友好） |
| 验证脚本 | `verify_lineage.py` | `verify_lineage.py`（manual）+ `verify_auto_lineage.py`（auto） |

**auto 通道不替代 manual 通道**，是并行运行：auto 覆盖 ETL 真实派生边，manual 覆盖业务跨系统 JOIN。

## 步骤 1：加载 manual 通道（lineage_recipe.yaml）

用 module3 的 `dg_education.lineage` API 加载手工 recipe，构建「manual 边」图。

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path
from dg_education.lineage import load_lineage_graph, render_ascii

RECIPE_PATH = Path('..') / 'lineage_recipe.yaml'
graph = load_lineage_graph(RECIPE_PATH)
print(f'manual 边数: {graph.number_of_edges()}')
print(f'manual 节点数: {graph.number_of_nodes()}')
print()
print(render_ascii(graph))

## 步骤 2：触发 manual 通道（emit_lineage.py）

把 recipe 里的 8 条边写进 GMS，供后续 visual 对比。

In [ ]:
import subprocess
result = subprocess.run(
    ['uv', 'run', 'python', 'scripts/emit_lineage.py'],
    capture_output=True, text=True, cwd='..',
)
print(result.stdout[-1500:])

## 步骤 3：触发 auto 通道（LineageEmitter）

ETL 入口插入 `with LineageEmitter(...) as e: e.emit_output(...)` 即可。
这里手动模拟一次 emit（生产中由 ETL 任务自动触发）。

选 `dwd_vbak` 这个目标作为 auto 边代表：上游 `sap_erp.vbak`、下游 DWA/DWD。

In [ ]:
import sys
sys.path.insert(0, '..')
from dg_platform.lineage_emitter import LineageEmitter

OUTPUT_URN = 'urn:li:dataset:(urn:li:dataPlatform:sap_erp,dwd_vbak,PROD)'
SQL = 'SELECT * FROM sap_erp.vbak WHERE ERDAT IS NOT NULL'

with LineageEmitter('ingest_dwd', sql=SQL, output_urn=OUTPUT_URN) as e:
    e.emit_output(OUTPUT_URN)
    print(f'emit started, run_id={e._run_id} (len={len(e._run_id)})')
print('emit completed (COMPLETE event sent to GMS OpenLineage endpoint)')

## 步骤 4：GMS 真值查询（auto vs manual 双通道）

对同一目标 dataset `dwd_vbak`，查 GMS 中是否有 auto 通道的 dataJob + manual 通道的 upstreamLineage。

In [ ]:
import time, urllib.parse, requests

GMS = 'http://localhost:28080'
AUTH = ('datahub', 'datahub')

# 等待 GMS Kafka 异步消费落库
time.sleep(5)

# 1. Manual 通道：查 dataset 的 upstreamLineage aspect
URN = 'urn:li:dataset:(urn:li:dataPlatform:sap_erp,dwd_vbak,PROD)'
ENC = urllib.parse.quote(URN, safe='')
r = requests.get(f'{GMS}/aspects/{ENC}?aspect=upstreamLineage&version=0', auth=AUTH)
print(f'manual 通道 upstreamLineage: HTTP {r.status_code}')
if r.status_code == 200:
    data = r.json().get('aspect', {}).get('com.linkedin.dataset.UpstreamLineage', {})
    upstreams = [u['dataset'] for u in data.get('upstreams', [])]
    print(f'  upstreams: {upstreams}')

# 2. Auto 通道：查 dataJob 的 dataJobInputOutput aspect
JOB_URN = 'urn:li:dataJob:(urn:li:dataFlow:(python,ingest_dwd,dg-demo),ingest_dwd)'
JOB_ENC = urllib.parse.quote(JOB_URN, safe='')
r2 = requests.get(f'{GMS}/aspects/{JOB_ENC}?aspect=dataJobInputOutput&version=0', auth=AUTH)
print(f'auto 通道 dataJobInputOutput: HTTP {r2.status_code}')
if r2.status_code == 200:
    data = r2.json().get('aspect', {}).get('com.linkedin.datajob.DataJobInputOutput', {})
    inputs = [e.get('destinationUrn', '') for e in data.get('inputDatasetEdges', [])]
    outputs = [e.get('destinationUrn', '') for e in data.get('outputDatasetEdges', [])]
    print(f'  inputs: {inputs}')
    print(f'  outputs: {outputs}')

## 步骤 5：跑 verify_auto_lineage.py 端到端验证

封装好的脚本：检查所有 7 条 auto 边的 dataJob 落库 + OpenSearch 索引同步。

In [ ]:
import subprocess
r = subprocess.run(
    ['uv', 'run', 'python', 'scripts/verify_auto_lineage.py'],
    capture_output=True, text=True, cwd='..',
)
print(r.stdout)
print('---stderr---')
print(r.stderr)
print(f'exit code: {r.returncode}')

## 步骤 6：DataHub UI 视觉对比（Playwright 截图）

Open `http://localhost:29002`，对 `sap_erp.dwd_vbak`（auto 边目标）和 `lims.samples`（manual 边代表）
分别抓 Lineage 标签页的截图。

截图会保存到 `notebook/step_images/module9_lineage_auto.png` 与 `module9_lineage_manual.png`。

In [ ]:
import time
import urllib.parse
from pathlib import Path
from playwright.sync_api import sync_playwright

UI = 'http://localhost:29002'
OUT_DIR = Path('step_images')
OUT_DIR.mkdir(parents=True, exist_ok=True)

time.sleep(5)  # 等 OpenSearch 索引刷新

URL_AUTO = f"{UI}/dataset/{urllib.parse.quote('urn:li:dataset:(urn:li:dataPlatform:sap_erp,dwd_vbak,PROD)', safe='')}"
URL_MANUAL = f"{UI}/dataset/{urllib.parse.quote('urn:li:dataset:(urn:li:dataPlatform:lims,samples,PROD)', safe='')}"

def _capture(page, url, out_path):
    page.goto(url, wait_until='domcontentloaded', timeout=60000)
    page.wait_for_timeout(6000)
    for sel in ["role=tab[name='Lineage']", "text=Lineage"]:
        try:
            page.locator(sel).first.click(timeout=8000)
            break
        except Exception:
            continue
    page.wait_for_timeout(4000)
    page.screenshot(path=str(out_path), full_page=True)

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    ctx = browser.new_context(viewport={'width': 1600, 'height': 1000})
    page = ctx.new_page()
    _capture(page, URL_MANUAL, OUT_DIR / 'module9_lineage_manual.png')
    print(f'saved: {OUT_DIR / "module9_lineage_manual.png"}')
    _capture(page, URL_AUTO, OUT_DIR / 'module9_lineage_auto.png')
    print(f'saved: {OUT_DIR / "module9_lineage_auto.png"}')
    browser.close()

## 步骤 7：双通道去重验证

DataHub UI 的 lineage 图按 `(dataset, upstream)` 唯一键去重——manual 边和 auto 边指向同一对时只显示 1 条。

本项目 `dwd.vbak ← sap_erp.vbak` 这一对：manual 通过 `lineage_recipe.yaml` 写，auto 通过 `ingest_dwd` emit。
DataHub graph 服务会合并为 1 条边。

In [ ]:
import requests
URN = 'urn:li:dataset:(urn:li:dataPlatform:dwd,vbak,PROD)'
ENC = urllib.parse.quote(URN, safe='')
r = requests.get(
    f'http://localhost:28080/openapi/relationships/v1',
    params={'urn': URN, 'relationshipTypes': 'DownstreamOf', 'direction': 'INCOMING'},
    auth=('datahub', 'datahub'),
)
print(f'查询 dwd.vbak 上游: HTTP {r.status_code}, total={r.json().get("total", 0)}')

## 总结

- **auto 通道**：ETL 任务 `with LineageEmitter(...)` 上下文管理器，零 YAML 维护。
- **manual 通道**：`lineage_recipe.yaml` + `emit_lineage.py`，继续兜底跨系统业务 JOIN。
- **并存**：DataHub graph 服务按 `(dataset, upstream)` 唯一键去重，UI 不重复。
- **延迟**：auto 通道 emit 完成后 ≤ 30s 在 UI 可见（实测 ~5s）。

详细见 `docs/Module3.md` 第 6.9 节、`docs/Background.md` §6.3 升级说明。